# EDA Workshop — TechData Store Q1 2024 Sales

**Dataset:** `datasets/sales.csv` — 54 transactions across 5 regions (Jan–Mar 2024)

**Goal:** Answer the question: *Which product categories and regions drive the most revenue?*

---
### Steps
1. Load & Inspect
2. Data Cleaning
3. Univariate Analysis
4. Revenue by Category & Region
5. Monthly Trend
6. Top Products
7. Key Findings

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Display settings
pd.set_option('display.float_format', '{:,.0f}'.format)
sns.set_theme(style='whitegrid')
%matplotlib inline

DATASETS = os.path.join(os.path.abspath(''), '../../../datasets')
print('Setup complete')

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(os.path.join(DATASETS, 'sales.csv'), parse_dates=['date'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\n--- Missing values ---')
print(df.isnull().sum())

In [ ]:
df.describe()

## 2. Data Cleaning

In [ ]:
# Add useful derived columns
df['month'] = df['date'].dt.strftime('%Y-%m')
df['month_name'] = df['date'].dt.strftime('%b')

# Verify revenue = quantity * unit_price
df['expected_revenue'] = df['quantity'] * df['unit_price']
mismatches = df[abs(df['revenue'] - df['expected_revenue']) > 0.01]
print(f'Revenue mismatches: {len(mismatches)}')
df.drop(columns=['expected_revenue'], inplace=True)

print('Unique categories:', df['category'].unique())
print('Unique regions:   ', df['region'].unique())

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['revenue'].hist(bins=15, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Revenue Distribution')
axes[0].set_xlabel('Revenue (THB)')

df['quantity'].hist(bins=10, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Quantity Distribution')
axes[1].set_xlabel('Units Sold')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Transactions by Category')
axes[0].tick_params(axis='x', rotation=30)

df['region'].value_counts().plot(kind='bar', ax=axes[1], color='mediumseagreen')
axes[1].set_title('Transactions by Region')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Revenue by Category & Region

In [ ]:
by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print('Revenue by Category:')
print(by_cat)

by_cat.plot(kind='bar', figsize=(8, 4), color='steelblue', edgecolor='white')
plt.title('Total Revenue by Category')
plt.ylabel('Revenue (THB)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
by_region = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
print('Revenue by Region:')
print(by_region)

by_region.plot(kind='barh', figsize=(8, 4), color='mediumseagreen')
plt.title('Total Revenue by Region')
plt.xlabel('Revenue (THB)')
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(values='revenue', index='region', columns='category', aggfunc='sum', fill_value=0)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt=',', cmap='YlOrRd')
plt.title('Revenue Heatmap: Region vs Category')
plt.tight_layout()
plt.show()

## 5. Monthly Revenue Trend

In [ ]:
monthly = df.groupby('month')['revenue'].sum()
print(monthly)

monthly.plot(kind='line', marker='o', figsize=(8, 4), color='steelblue', linewidth=2)
plt.title('Monthly Revenue Trend — Q1 2024')
plt.ylabel('Revenue (THB)')
plt.xlabel('Month')
plt.tight_layout()
plt.show()

In [ ]:
monthly_by_cat = df.groupby(['month', 'category'])['revenue'].sum().unstack(fill_value=0)
monthly_by_cat.plot(kind='bar', stacked=True, figsize=(10, 5))
plt.title('Monthly Revenue by Category')
plt.ylabel('Revenue (THB)')
plt.xticks(rotation=0)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Top Products

In [ ]:
top_products = (
    df.groupby('product')
    .agg(total_revenue=('revenue', 'sum'), total_qty=('quantity', 'sum'), txn_count=('revenue', 'count'))
    .sort_values('total_revenue', ascending=False)
)
print(top_products)

In [ ]:
top_products['total_revenue'].plot(kind='barh', figsize=(9, 6), color='coral')
plt.title('Total Revenue by Product')
plt.xlabel('Revenue (THB)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Key Findings

In [ ]:
total_rev = df['revenue'].sum()
top_cat = by_cat.idxmax()
top_region = by_region.idxmax()
top_product = top_products['total_revenue'].idxmax()
best_month = monthly.idxmax()

print('=' * 50)
print('KEY FINDINGS — Q1 2024')
print('=' * 50)
print(f'Total Revenue    : {total_rev:>12,.0f} THB')
print(f'Top Category     : {top_cat} ({by_cat[top_cat] / total_rev * 100:.1f}% of revenue)')
print(f'Top Region       : {top_region} ({by_region[top_region] / total_rev * 100:.1f}% of revenue)')
print(f'Top Product      : {top_product}')
print(f'Best Month       : {best_month}')
print(f'Growth Jan->Mar  : +{(monthly.iloc[-1] - monthly.iloc[0]) / monthly.iloc[0] * 100:.1f}%')

---
## Try It Yourself

1. Which region has the highest average revenue **per transaction**?
2. Which **day of the week** has the most sales?
3. Is there a product that appears in every region?
4. What is the month-over-month growth rate by category?

**Hint:** Use `df['date'].dt.day_name()` for day of week.